# 2. Reserva del conjunto de prueba

Antes de cualquier decisión que se use en el modelado (imputación, transformaciones, selección de variables, umbrales de atípicos o hiperparámetros), se reserva el conjunto de prueba. Primero se eliminan los duplicados exactos detectados en la sección 1.6.3; es una limpieza estructural que no usa estadísticas de los datos y conserva el orden de las filas.

**Partición cronológica 80/20.** El entrenamiento es el primer 80 % de los registros en orden temporal y la prueba, el último 20 %. La decisión responde a la estructura de los datos y al propósito del modelo:

- Las filas están ordenadas por fecha (sección 1.5) y el enunciado exige una partición cronológica cuando hay tiempo.
- El modelo debe decidir a quién llamar en **campañas futuras**, así que la prueba debe simular ese uso: ajustar con el pasado y evaluar con periodos posteriores que el modelo no vio.
- La sección 3.6 confirma, solo con datos de entrenamiento, que una validación aleatoria inflaría el desempeño (**fuga temporal**): un modelo que únicamente reconoce el periodo alcanza casi el mismo AUC que el modelo completo, y la validación temporal da un resultado mucho menor.

La partición no se estratifica: la prevalencia de la clase positiva cambia entre periodos y se reporta tal como es, porque forzarla a ser igual ocultaría justamente el cambio que el modelo enfrentará en la práctica. Desde aquí, **todo el análisis exploratorio y todas las decisiones se calculan solo con el conjunto de entrenamiento**. El conjunto de prueba se usa una única vez, en la evaluación final de la sección 4.4.

In [11]:
df_limpio = df.drop_duplicates(keep="first")
assert df_limpio.index.is_monotonic_increasing  # se conserva el orden temporal del archivo
y_todo = df_limpio[OBJETIVO].eq("yes").astype(int)
X_todo = df_limpio.drop(columns=OBJETIVO)

n_prueba = int(np.ceil(0.20 * len(X_todo)))
n_entrena = len(X_todo) - n_prueba
X_tr, X_te = X_todo.iloc[:n_entrena], X_todo.iloc[n_entrena:]
y_tr, y_te = y_todo.iloc[:n_entrena], y_todo.iloc[n_entrena:]
anio_tr, anio_te = tiempo.loc[X_tr.index, "anio"], tiempo.loc[X_te.index, "anio"]
periodo_tr, periodo_te = tiempo.loc[X_tr.index, "periodo"], tiempo.loc[X_te.index, "periodo"]
dia_tr, dia_te = tiempo.loc[X_tr.index, "dia"], tiempo.loc[X_te.index, "dia"]
tr = X_tr.assign(y=y_tr)  # copia de trabajo para el EDA (solo entrenamiento)
periodo_corte = periodo_tr.iloc[-1]
dias_compartidos = len(set(dia_tr) & set(dia_te))

particion = pd.DataFrame({
    "Filas": [len(df_limpio), len(X_tr), len(X_te)],
    "Desde": [tiempo.loc[X_todo.index, "periodo"].iloc[0], periodo_tr.iloc[0], periodo_te.iloc[0]],
    "Hasta": [tiempo.loc[X_todo.index, "periodo"].iloc[-1], periodo_tr.iloc[-1], periodo_te.iloc[-1]],
    "Positivos (suscribe)": [int(y_todo.sum()), int(y_tr.sum()), int(y_te.sum())],
    "% positivos": [100 * y_todo.mean(), 100 * y_tr.mean(), 100 * y_te.mean()],
    "% de 2008": [100 * (tiempo.loc[X_todo.index, "anio"] == 2008).mean(), 100 * (anio_tr == 2008).mean(),
                  100 * (anio_te == 2008).mean()],
    "% de 2009": [100 * (tiempo.loc[X_todo.index, "anio"] == 2009).mean(), 100 * (anio_tr == 2009).mean(),
                  100 * (anio_te == 2009).mean()],
    "% de 2010": [100 * (tiempo.loc[X_todo.index, "anio"] == 2010).mean(), 100 * (anio_tr == 2010).mean(),
                  100 * (anio_te == 2010).mean()]},
    index=["Total sin duplicados exactos", "Entrenamiento (primer 80 %)", "Prueba (último 20 %)"])
tabla(particion, dec=2)

interpretar(
    f"Tras eliminar {len(df) - len(df_limpio)} duplicados exactos quedan {num(len(df_limpio))} registros. El "
    f"entrenamiento tiene {num(len(X_tr))} registros, de {periodo_tr.iloc[0]} a {periodo_tr.iloc[-1]}, y la prueba "
    f"{num(len(X_te))}, de {periodo_te.iloc[0]} a {periodo_te.iloc[-1]}. Suscribe el {pct(y_tr.mean())} en "
    f"entrenamiento y el {pct(y_te.mean())} en prueba, {num(y_te.mean() / y_tr.mean(), 1)} veces más. El "
    f"entrenamiento es en un {pct((anio_tr == 2008).mean())} de 2008, mientras que la prueba es en un "
    f"{pct((anio_te == 2010).mean())} de 2010 y no tiene registros de 2008. El corte cae dentro de {periodo_corte}, "
    f"único mes presente en ambos conjuntos"
    + (f" (comparten {dias_compartidos} día de contacto)." if dias_compartidos == 1 else
       (f" (comparten {dias_compartidos} días de contacto)." if dias_compartidos > 1 else ".")),
    "De esta forma, la prueba mide lo que importa para el uso real: el desempeño en periodos posteriores a los del "
    "ajuste. El cambio de prevalencia es un desplazamiento de la clase (*prior shift*) que el modelo no observa "
    f"durante el ajuste: las probabilidades aprendidas con una tasa base del {pct(y_tr.mean())} tenderán a subestimar "
    "la de prueba, y un umbral elegido en entrenamiento puede no servir en prueba. También cambia la línea base: el "
    f"Dummy, que siempre predice \"no suscribe\", acierta en prueba el {pct(1 - y_te.mean())} y no el "
    f"{pct(1 - y_tr.mean())} que lograría en entrenamiento.",
    "Por tanto, corresponde validar dentro de entrenamiento con un esquema temporal (sección 3), evaluar con métricas "
    "que no dependan del umbral (AUC) junto con AUC-PR, recall, F1 y calibración, y analizar en la sección 4 el efecto "
    "del cambio de prevalencia sobre el umbral y la calibración.")

,Filas,Desde,Hasta,Positivos (suscribe),% positivos,% de 2008,% de 2009,% de 2010
Total sin duplicados exactos,41 176,2008-05,2010-11,4 639,"11,27","67,23","27,77","5,00"
Entrenamiento (primer 80 %),32 940,2008-05,2009-05,2 100,"6,38","84,04","15,96","0,00"
Prueba (último 20 %),8 236,2009-05,2010-11,2 539,"30,83","0,00","75,01","24,99"


**Interpretación.** Tras eliminar 12 duplicados exactos quedan 41 176 registros. El entrenamiento tiene 32 940 registros, de 2008-05 a 2009-05, y la prueba 8 236, de 2009-05 a 2010-11. Suscribe el 6,4 % en entrenamiento y el 30,8 % en prueba, 4,8 veces más. El entrenamiento es en un 84,0 % de 2008, mientras que la prueba es en un 25,0 % de 2010 y no tiene registros de 2008. El corte cae dentro de 2009-05, único mes presente en ambos conjuntos (comparten 1 día de contacto).

De esta forma, la prueba mide lo que importa para el uso real: el desempeño en periodos posteriores a los del ajuste. El cambio de prevalencia es un desplazamiento de la clase (*prior shift*) que el modelo no observa durante el ajuste: las probabilidades aprendidas con una tasa base del 6,4 % tenderán a subestimar la de prueba, y un umbral elegido en entrenamiento puede no servir en prueba. También cambia la línea base: el Dummy, que siempre predice "no suscribe", acierta en prueba el 69,2 % y no el 93,6 % que lograría en entrenamiento.

Por tanto, corresponde validar dentro de entrenamiento con un esquema temporal (sección 3), evaluar con métricas que no dependan del umbral (AUC) junto con AUC-PR, recall, F1 y calibración, y analizar en la sección 4 el efecto del cambio de prevalencia sobre el umbral y la calibración.